# CS4765/6765 NLP Assignment 3: Word vectors

**Due 28 October**

In this two part assignment you will first examine and interact with word vectors. (This part of the assignment is adapted from a CS224N assignment at Stanford.) You will then implement a new approach to sentiment analysis.

In this assignment we will use [gensim](https://radimrehurek.com/gensim/) to access and interact with word embeddings. In gensim we’ll be working with a KeyedVectors object which represents word embeddings. [Documentation for KeyedVectors is available.](https://radimrehurek.com/gensim/models/keyedvectors.html) However, this assignment description and the sample code in it might be sufficient to show you how to use a KeyedVectors object. We will use [GloVe word embeddings](https://nlp.stanford.edu/projects/glove/) that have been trained on Wikipedia and the Gigaword corpus.


In [1]:
import gensim.downloader
model = gensim.downloader.load('glove-wiki-gigaword-300')

# Part 1: Examining word vectors (8 marks)

## Polysemy and homonymy

Polysemy and homonymy are the phenomena of words having multiple meanings/senses. The nearest neighbours (under cosine similarity) for a given word can indicate whether it has multiple senses.

Consider the following example which shows the top-10 most similar words for *mouse*. The "input device" and "animal" senses of *mouse* are clearly visible from the top-10 most similar words. 


In [2]:
# Find words most similar using cosine similarity to "mouse". 
# restrict_vocab=100000 limits the results to most frequent
# 100000 words. This avoids rare words in the output. For this
# assignment, whenever you call most_similar, also pass
# restrict_vocab=100000.
model.most_similar('mouse', restrict_vocab=100000)

[('mice', 0.6210126876831055),
 ('rat', 0.5267991423606873),
 ('keyboard', 0.5248469114303589),
 ('rabbit', 0.5081881880760193),
 ('rodent', 0.49729210138320923),
 ('monkey', 0.4925020635128021),
 ('joystick', 0.4715430736541748),
 ('rats', 0.4617359936237335),
 ('cursor', 0.4608822166919708),
 ('cat', 0.4537910223007202)]

*keyboard*, *joystick*, and *cursor* correspond to the input device sense. *mice*, *rat*, *rabbit*, *rodent*, *monkey*, *rats*, and *cat* correspond to the animal sense. (You can observe something similar for the different senses of the word *leaves*.)

Find a new example that exhibits polysemy/homonymy, show its top-10 most similar words, and explain why they show that this word has multiple senses. Write your answer in the code and text boxes below.

In [3]:
model.most_similar("plant", restrict_vocab=100000)

[('plants', 0.8378477096557617),
 ('factory', 0.575928270816803),
 ('facility', 0.5050454139709473),
 ('flowering', 0.485045850276947),
 ('produce', 0.47375598549842834),
 ('reactor', 0.4689636528491974),
 ('factories', 0.46837306022644043),
 ('production', 0.46605584025382996),
 ('manufacturing', 0.4617835283279419),
 ('producing', 0.4426949620246887)]

The word `plant` can refer to a `living organism` or an `industrial facility`. The neighbour `flowering` relates to `living plants`, while `factory`, `factories`, `manufacturing`, and `reactor` relate to industrial facilities. Both senses appear in the top 10 because the model represents the word with one vector that reflects its different contexts.

## Synonyms and antonyms

Find three words (w1 , w2 , w3) such that w1 and w2 are synonyms (i.e., have roughly the same meaning), and w1 and w3 are antonyms (i.e., have opposite meanings), but the similarity between w1 and w3 > the similarity between w1 and w2. Note that this should be counter to your expectations, because synonyms (which mean roughly the same thing) would be expected to be more similar than antonyms (which have opposite meanings). Explain why you think this unexpected situation might have occurred.

Here is an example. w1 = *happy*, w2 = *cheerful*, and w3 = *sad*. (You will need to find a different example for your answer.) Notice that the antonyms *happy* and *sad* are more similar than the (near) synonyms *happy* and *cheerful*.


In [4]:
# Find the cosine similarity between "happy" and "cheerful"
model.similarity('happy', 'cheerful')

0.44031656

In [5]:
# and between "happy" and "sad".
model.similarity('happy', 'sad')

0.5652857

In [6]:
print("strong and powerful:", model.similarity('strong', 'powerful'))
print("strong and weak:", model.similarity('strong', 'weak'))

strong and powerful: 0.54768085
strong and weak: 0.62324786


`strong` and `powerful` are near synonyms, while `strong` and `weak` are antonyms. However, the similarity between strong and weak is 0.6232, which is higher than 0.5477 for strong and powerful.

This can happen because embeddings reflect the contexts in which words occur. Both strong and weak can describe arguments, evidence, or economies. These shared contexts can produce similar vectors despite their opposite meanings.

## Analogies

Analogies such as man is to king as woman is to X can be solved using word embeddings. This analogy can be expressed as X = woman + king − man. The following code snippet shows how to solve this analogy with gensim. Notice that the model gets it correct! I.e., *queen* is the most similar word.

In [7]:
# Find the model's predictions for the solution to the analogy
# "man" is to "king" as "woman" is to X
model.most_similar(positive=['woman', 'king'],
                   negative=['man'],
                   restrict_vocab=100000)

[('queen', 0.6713277101516724),
 ('princess', 0.5432624220848083),
 ('throne', 0.5386105179786682),
 ('monarch', 0.5347574949264526),
 ('daughter', 0.4980250597000122),
 ('mother', 0.4956442713737488),
 ('elizabeth', 0.4832652509212494),
 ('kingdom', 0.47747090458869934),
 ('prince', 0.4668240249156952),
 ('wife', 0.46473270654678345)]

### Correct analogy

Find a new analogy that the model is able to answer correctly (i.e., the most-similar word is the solution to the analogy). Explain briefly why the analogy holds. For the above example, this explanation would be something along the lines of a king is a ruler who is a man and a queen is a ruler who is a woman.


In [8]:
# pakistan : urdu :: france : french
model.most_similar(positive=['urdu', 'france'],
                   negative=['pakistan'],
                   restrict_vocab=100000)

[('french', 0.5092459321022034),
 ('arabic', 0.4554755687713623),
 ('occitan', 0.4528762996196747),
 ('yiddish', 0.4215300977230072),
 ('hindi', 0.4135982394218445),
 ('translated', 0.41016653180122375),
 ('marathi', 0.4017965495586395),
 ('english', 0.3988052010536194),
 ('afrikaans', 0.39740434288978577),
 ('language', 0.39552417397499084)]

The analogy is `Pakistan` is to `Urdu` as `France` is to `French`. Urdu is Pakistan’s national language, and French is France’s official language. The model returns french as the most similar word, so it answers this analogy correctly.

### Incorrect analogy

Find a new analogy that the model is not able to answer correctly. Again explain briefly why the analogy holds. For example, here is an analogy that the model does not answer correctly:


In [9]:
# Find the model's predictions for the solution to the analogy
# "finger" is to "hand" as "toe" is to X
model.most_similar(positive=['toe', 'hand'],
                   negative=['finger'],
                   restrict_vocab=100000)

[('boots', 0.45490795373916626),
 ('hands', 0.45022207498550415),
 ('shoes', 0.44836610555648804),
 ('wear', 0.44443702697753906),
 ('right', 0.4407408833503723),
 ('wearing', 0.4199027419090271),
 ('shoulder', 0.40702393651008606),
 ('back', 0.40581828355789185),
 ('legs', 0.4050144851207733),
 ('put', 0.4037577509880066)]

A finger is part of a hand, and a toe is part of a foot, but the model does not predict *foot*, or a similar term, as the most similar word.

In [10]:
# Cow is to calf as sheep is to lamb.
model.most_similar(positive=['sheep', 'calf'],
                   negative=['cow'],
                   restrict_vocab=100000)

[('groin', 0.5216478705406189),
 ('thigh', 0.5202720165252686),
 ('hamstring', 0.5073035955429077),
 ('calves', 0.4950656592845917),
 ('goats', 0.46974197030067444),
 ('ankle', 0.4670944809913635),
 ('quadriceps', 0.45185214281082153),
 ('knee', 0.4467581808567047),
 ('ligaments', 0.43656039237976074),
 ('tendon', 0.42125996947288513)]

The analogy is `Cow` is to `calf` as `sheep` is to `lamb`. A calf is a young cow, and a lamb is a young sheep. The expected answer is `lamb`, but the model returns `groin` as the most similar word.

Other results include `thigh` and `hamstring`. It suggests that the leg-related sense of `calf` may affect the result.

## Bias

Consider the examples below. The first shows the words that are most similar to *man* and *worker* and least similar to *woman*. The second shows the words that are most similar to *woman* and *worker* and least similar to *man*.

In [11]:
# Find the words that are most similar to "man" and "worker" and
# least similar to "woman".
model.most_similar(positive=['man', 'worker'],
                   negative=['woman'],
                   restrict_vocab=100000)

[('workers', 0.5640615820884705),
 ('employee', 0.5365461707115173),
 ('laborer', 0.48308447003364563),
 ('working', 0.4746786653995514),
 ('factory', 0.4493158757686615),
 ('mechanic', 0.43802666664123535),
 ('work', 0.4276600480079651),
 ('unemployed', 0.42742663621902466),
 ('worked', 0.4222966134548187),
 ('job', 0.42074185609817505)]

In [12]:
# Find the words that are most similar to "woman" and "worker" and
# least similar to "man".
model.most_similar(positive=['woman', 'worker'],
                   negative=['man'],
                   restrict_vocab=100000)

[('employee', 0.5915157794952393),
 ('workers', 0.5560789108276367),
 ('nurse', 0.514857828617096),
 ('pregnant', 0.4897522032260895),
 ('mother', 0.48388367891311646),
 ('female', 0.46243950724601746),
 ('child', 0.4448588192462921),
 ('teacher', 0.44152435660362244),
 ('waitress', 0.44121506810188293),
 ('employer', 0.4378712773323059)]

The output shows that *man* is associated with some stereotypically male jobs (e.g., *mechanic*) while *woman* is associated with some stereotypically female jobs (e.g., *nurse*). This indicates that there is gender bias in the word embeddings.

Find a new example, using the same approach as above, that indicates that there is bias in the word embeddings. Briefly explain how the model output indicates that there is bias in the word embeddings. (You are by no means restricted to considering gender bias here. You are encouraged to explore other ways that embeddings might indicate bias.)

In [13]:
model.most_similar(positive=['man', 'brilliant'],
                   negative=['woman'],
                   restrict_vocab=100000)

[('superb', 0.5880734920501709),
 ('brilliantly', 0.5206077098846436),
 ('genius', 0.517926812171936),
 ('dazzling', 0.49841737747192383),
 ('spectacular', 0.4578637182712555),
 ('fantastic', 0.45781680941581726),
 ('clever', 0.45673123002052307),
 ('impressive', 0.4522416889667511),
 ('magnificent', 0.4517892003059387),
 ('masterful', 0.4479045569896698)]

In [14]:
model.most_similar(positive=['woman', 'brilliant'],
                   negative=['man'],
                   restrict_vocab=100000)

[('dazzling', 0.5670727491378784),
 ('superb', 0.5030891299247742),
 ('beautiful', 0.49297863245010376),
 ('gorgeous', 0.46522030234336853),
 ('bright', 0.45603764057159424),
 ('lovely', 0.44476720690727234),
 ('stunning', 0.44291383028030396),
 ('wonderful', 0.44155165553092957),
 ('brilliantly', 0.4347189962863922),
 ('gutsy', 0.43337440490722656)]

The query `man + brilliant − woman` returns words such as `genius`, `clever`, and `masterful`, which relate to intelligence or skill. 
The query `woman + brilliant − man` returns words such as `beautiful`, `gorgeous`, and `lovely`, which relate to appearance.

Although some words overlap, this difference suggests a gender stereotype associating men with intellectual ability and women with physical appearance. These associations reflect patterns learned from text, rather than actual differences in ability.

# Part 2: Sentiment Analysis (2 marks)

## Background and data

In this part of the assignment you will revisit sentiment analysis from assignment
2. You will need the data provided for that
assignment.


## Approach

We will consider sentiment analysis using an average of word embeddings document representation and a multinomial logistic regression classifier. We will compare this approach to the approach using tf-idf weighted unigram features and logistic regression from assignment 2.

Complete the function `vec_for_doc` below. (You should not modify other parts of the
code.) This function takes a list consisting of the tokens in a document $d$. It then returns a vector $\vec{v}$ representing the document as the average of the embeddings for the words in the document as follows:

\begin{equation}
d = w_1, w_2, ... w_n
\end{equation}
\begin{equation}
\vec{v} = \dfrac{\vec{w_1} + \vec{w_2} + ... + \vec{w_n}}{n}\\
\end{equation}

If a word in a document does not occur in the word embedding model, you can simply ignore it. As such, $n$ above is the number of token instances in document $d$ that also occur in the embedding model (as opposed to simply the number of token instances in document $d$). (Note that we would normally need to deal with the case of a document that consists entirely of words that don't occur in the embedding model, but for this dataset and embedding model, that situation does not occur, and so for now we don't worry about it.)

In [15]:
import math
def normalize(v):
    return v/math.sqrt(sum(v**2))

# TODO: Implement this function. tokenized_doc is a list of tokens in
# a document. Return a vector representation of the document as
# described above.
# Hints: 
# -You can get the vector for a word w using model[w] or
#  model.get_vector(w)
# -You can add vectors using + and sum, e.g.,
#  model['cat'] + model['dog']
#  sum([model['cat'], model['dog']])
# -You can see the shape of a vector using model['cat'].shape
# -The vector you return should have the same shape as a word vector 
# -This should be a very short function. If you're writing lots of
#  code, you are likely off track.
def vec_for_doc(tokenized_doc):
    word_vectors = []
    for token in tokenized_doc:
        if token in model:
            word_vectors.append(model[token])
    return sum(word_vectors) / len(word_vectors)
    

Once you've completed `vec_for_doc` above, run the code below to train logistic regresion on the training data and evaluate on the dev data

In [16]:
# This code is taken from the assignment 2 starter code
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import csv

def get_texts_and_labels(fname):
    # Added argument for encoding
    csv_reader = csv.reader(open(fname, encoding='utf8'))
    # Ignore header row
    next(csv_reader)
    texts = []
    labels = []
    for line in csv_reader:
        _,text,label = line
        label = int(label)
        texts.append(text)
        labels.append(label)
    return texts,labels

def get_results(gold_labels, predicted_labels):
    p,r,f,_ = precision_recall_fscore_support(gold_labels, 
                                              predicted_labels,
                                              average='macro',
                                              zero_division=0)
    acc = accuracy_score(gold_labels, predicted_labels)
    return p,r,f,acc    

def print_results(gold_labels, predicted_labels):
    p,r,f,acc = get_results(gold_labels, predicted_labels)
    
    print("Precision: ", p)
    print("Recall: ", r)
    print("F1: ", f)
    print("Accuracy: ", acc)
    print()

train_texts, train_labels = get_texts_and_labels('data/train-sample.csv')
dev_texts, dev_labels = get_texts_and_labels('data/dev.csv')

In [17]:
# A simple tokenizer based on the word tokenizer from sklearn.
# It applies case folding. This should give the same tokenization
# as used by default by the sklearn word tokenizer.
import re
word_tokenize_pattern = re.compile(r"(?u)\b\w\w+\b")
def word_tokenize(s, apply_case_folding=True):
    return [x.lower() for x in word_tokenize_pattern.findall(s)]

# train_vecs and dev_vecs are lists; each element is a vector
# representing a (train or dev) document
# (These two lines are new, i.e., not from the assignment 2 starter code)
train_vecs = [vec_for_doc(word_tokenize(x)) for x in train_texts]
dev_vecs = [vec_for_doc(word_tokenize(x)) for x in dev_texts]

# Train logistic regression, same as for A2, except that max_iter
# is set to 500 so that training converges
lr = LogisticRegression(max_iter=500,
                        random_state=0)
clf = lr.fit(train_vecs, train_labels)
dev_predictions = clf.predict(dev_vecs)

print_results(dev_labels, dev_predictions)

Precision:  0.7641232575201761
Recall:  0.7607233440639938
F1:  0.7623477297895902
Accuracy:  0.765



Finally, evaluate on the test data

In [18]:
test_texts,test_labels = get_texts_and_labels('data/test.csv')
test_vecs = [vec_for_doc(word_tokenize(x)) for x in test_texts]
test_predictions = clf.predict(test_vecs)
print_results(test_labels, test_predictions)

Precision:  0.744365297617532
Recall:  0.7551198509268274
F1:  0.749045318610536
Accuracy:  0.78125



Run the code below to train a logistic regression classifier with tf-idf weighted unigram features and evaluate it on the test data. (This should be similar to what you did in assignment 2. When making comparisons between models (see below) use the results here, as opposed to your results from assignment 2.)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
train_tfidfs = tfidf_vectorizer.fit_transform(train_texts)
lr_tfidf = LogisticRegression(random_state=0)
lr_tfidf_classifier = lr_tfidf.fit(train_tfidfs, train_labels)

dev_tfidfs = tfidf_vectorizer.transform(dev_texts)
test_tfidfs = tfidf_vectorizer.transform(test_texts)

print("Tf-idf weighted unigram features")
print("Dev")
lr_tfidf_dev_predictions = lr_tfidf_classifier.predict(dev_tfidfs)
print_results(dev_labels, lr_tfidf_dev_predictions)

print("Test")
lr_tfidf_test_predictions = lr_tfidf_classifier.predict(test_tfidfs)
print_results(test_labels, lr_tfidf_test_predictions)

Tf-idf weighted unigram features
Dev
Precision:  0.8240139734054542
Recall:  0.7850136572041753
F1:  0.7987237076648842
Accuracy:  0.8

Test
Precision:  0.6965208564045774
Recall:  0.6674088549646041
F1:  0.6766528005338697
Accuracy:  0.73125



Compare the results on the test data using logistic regression with the average of word embeddings document representation to the results above using logistic regression with tf-idf weighted unigram features. The difference between these two approaches is the document representation, i.e., average of word embeddings vs unigram (word) counts. Which method performs better?

Average word embeddings perform better on the **test data**. Their macro F1 is **0.7490** and accuracy is **78.125%**, compared with **0.6767** and **73.125%** for TF-IDF weighted unigrams.

As we know that the embeddings give similar representations to related words. This may help the classifier generalize when related words appear in different training and test documents. TF-IDF uses a separate feature for each word, so related words do not automatically share information.

On the development data, TF-IDF has the higher macro F1 (0.7987 versus 0.7623). Therefore, the advantage of averaged embeddings here is specific to the observed test results. But it does not mean they always perform better.

# Submitting your work

When you're done, please submit a3.ipynb to the assignment 3 folder on D2L.